# 06 · Evidence attribution, stability and reliability
### BeliefLens · STAC AI Workshop · NYC · 21 October 2026

## Learning objectives
1. Trace claims through a semantic graph
2. Run controlled source interventions
3. Measure reliability against labels
4. Distinguish stability, calibration and language probability

All companies, numbers, labels and market observations in these notebooks are synthetic teaching examples. Offline outputs come from explicit deterministic rules, not an LLM. The same JavaScript case engine runs in the browser and through this Python client. Only a cell explicitly using `live=True` calls Together AI. No financial action is executed.

**Requirements:** Python 3.10+, Node.js 20+, Jupyter. No model weights or API key are needed for the default exercises. Run cells in order.

In [ ]:
from pathlib import Path
import sys, json, math, random, statistics
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)
if root is None:
    raise RuntimeError("Extract the complete workshop ZIP and open Jupyter inside its folder.")
if str(root) not in sys.path: sys.path.insert(0, str(root))
from lab_client import run, compact, save
print("Shared engine ready:", compact(run()))

## Inspect the graph
The case template constructs source, tool, language, claim, rule and decision nodes. Dependencies are explicit; they are not a reconstruction of the model’s hidden computation. Expert-defined rules inspect particular nodes and gate the decision.

In [ ]:
baseline=run()
for node in baseline["graph"]["nodes"]:print(node["id"],node["type"],node["label"])
for edge in baseline["graph"]["edges"]:print(edge["from"],"--"+edge["label"]+"-->",edge["to"])
for change in [{"dropCovenant":True},{"futureLeak":True},{"wording":"rephrase"}]:
    r=run(change)
    print(change, compact(r))

## A small fault-injection benchmark
Count false releases against the explicit policy specification. This is a test of the bounded controller on six cases, not a statistical certification of the product or a model benchmark.

In [ ]:
cases=[
 ({"disruption":False,"approved":True},"RELEASE"),
 ({"disruption":False,"approved":True,"revoked":True},"HOLD"),
 ({"disruption":False,"approved":True,"futureLeak":True},"HOLD"),
 ({"disruption":False,"approved":True,"dropCovenant":True},"HOLD"),
 ({"disruption":False,"approved":True,"cash":0},"HOLD"),
 ({"disruption":True,"approved":True},"HOLD")]
observed=[run(c)["gate"] for c,_ in cases]
false_accepts=sum(pred=="RELEASE" and gold=="HOLD" for pred,(_,gold) in zip(observed,cases))
false_rejects=sum(pred=="HOLD" and gold=="RELEASE" for pred,(_,gold) in zip(observed,cases))
print("False accepts:",false_accepts,"False rejects:",false_rejects,"Cases:",len(cases))
assert false_accepts==0 and false_rejects==0

## Reliability requires labelled outcomes
Create synthetic predicted risks and observed binary outcomes. Fit bin-wise calibration on a calibration partition, then evaluate on a separate test partition. Neither the original scores nor the mapping below comes from a language model. Real labels would require adjudicated outcomes and a representative sampling design.

In [ ]:
rng=random.Random(22)
data=[]
for i in range(1000):
    score=rng.uniform(0,1)
    true_risk=0.1+0.6*score
    data.append((score,int(rng.random()<true_risk)))
calibration,test=data[:600],data[600:]
bucket=lambda p:min(4,int(p*5))
rates={b:statistics.mean(y for p,y in calibration if bucket(p)==b) for b in range(5)}
brier=lambda pairs:statistics.mean((p-y)**2 for p,y in pairs)
print("Uncalibrated Brier:",round(brier(test),4))
print("Mapped Brier:",round(brier([(rates[bucket(p)],y) for p,y in test]),4))
for b in range(5):
    sample=[(p,y) for p,y in test if bucket(p)==b]
    print("Bin",b,"n",len(sample),"mapped",round(rates[b],3),"observed",round(statistics.mean(y for _,y in sample),3))

## Learning check
1. The simulator returns the same answer ten times. Is this measured LLM stability? **No.**
2. A phrase has high token likelihood. Does that establish the claim’s truth probability? **No.**
3. A calibrated score on one population transfers automatically to a new population? **No.**

**Live extension:** repeat actual Together runs and rephrasings; record output variation separately from expert-labelled correctness. Source removal demonstrates dependence under a specific intervention, not unrestricted causal attribution.

In [ ]:
# OPTIONAL: execute only after starting the local server with Together configured.
# Never save credentials in a notebook cell or output.
# from getpass import getpass
# workshop_key = getpass("Temporary workshop key: ")
# actual = run({"mode": "rag_tools"}, live=True, key=workshop_key)
# print(compact(actual))
# print(actual.get("usage"), actual.get("latency_ms"))

## What you have demonstrated
- **Trace claims through a semantic graph:** Printed typed nodes and labelled edges
- **Run controlled source interventions:** Removed evidence and changed wording
- **Measure reliability against labels:** Computed fault-test errors and held-out Brier scores
- **Distinguish stability, calibration and language probability:** Separated scripted outputs, outcome calibration and model likelihood

**Exit ticket:** State one result, the evidence supporting it, and one limit that prevents you from claiming more. Save a run JSON or your numerical results as the experiment record.